# Development extension: data and protocol audit
Run from research/notebooks. This post-test extension cannot retrospectively improve the original final evaluation. No held-out features or labels are loaded.

In [1]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'research/src/supervised/final_holdout.py').exists())
BASE=ROOT/'data/processed/supervised/v1'
ART=BASE/'assignment_artifacts_20261030T020845905141Z'
OUT=ROOT/'data/processed/supervised/development_extension_v2'
OUT.mkdir(parents=True,exist_ok=True)
features=json.loads((ART/'artifact_manifest.json').read_text())['feature_allowlist']
df=pd.read_csv(ART/'development_train.csv',dtype={'recording_id':str})
assert len(df)==184 and df.recording_id.is_unique
assert len(features)==50 and np.isfinite(df[features].to_numpy()).all()
assert not set(features)&{'model_state','location_group','recording_id','scientific_name','latitude','longitude'}
X=df[features]; y=df.model_state; groups=df.location_group
CLASSES=['AZ','CA','TX']
assert set(y)==set(CLASSES) and groups.notna().all()
print('Development only:',len(df),'recordings;',groups.nunique(),'coordinate groups')

Development only: 184 recordings; 136 coordinate groups


In [2]:
from sklearn.model_selection import StratifiedGroupKFold
split=pd.read_csv(BASE/'splits/group_stratified_20261030T012037371263Z/primary_split_manifest.csv',dtype={'recording_id':str})
dev=split[split.partition.eq('development')]
assert set(dev.recording_id)==set(df.recording_id)
assert set(dev.location_group).isdisjoint(split.loc[split.partition.eq('test'),'location_group'])
aligned=dev.set_index('recording_id').loc[df.recording_id]
assert np.array_equal(aligned.model_state.to_numpy(),y.to_numpy())
assert np.array_equal(aligned.location_group.to_numpy(),groups.to_numpy())
display(df.groupby('model_state').agg(recordings=('recording_id','size'),locations=('location_group','nunique')))
display(aligned.groupby('model_state').scientific_name.nunique().rename('species'))
outer=list(StratifiedGroupKFold(5,shuffle=True,random_state=5612).split(X,y,groups))
assignment=[]
for fold,(tr,va) in enumerate(outer):
    assert set(groups.iloc[tr]).isdisjoint(groups.iloc[va])
    assert set(y.iloc[tr])==set(CLASSES) and set(y.iloc[va])==set(CLASSES)
    for i in va: assignment.append({'recording_id':df.recording_id.iloc[i],'outer_fold':fold})
pd.DataFrame(assignment).to_csv(OUT/'outer_folds.csv',index=False)
protocol={'purpose':'Exploratory post-test development extension; new independent test required',
 'selection':'inner 3-fold grouped CV macro-F1; outer 5-fold grouped CV',
 'seed':5612,'features':features,'development_sha256':hashlib.sha256((ART/'development_train.csv').read_bytes()).hexdigest(),
 'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__},
 'rules':['Do not reopen the old 52-recording test','Do not overwrite v1 results','No claim of improved final accuracy','Preprocessing fitted separately inside each inner training fold']}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Audit passed; outer folds and protocol saved.')

,recordings,locations
model_state,,
AZ,66,34
CA,65,56
TX,53,46


model_state
AZ    42
CA    45
TX    53
Name: species, dtype: int64

Audit passed; outer folds and protocol saved.
